# Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

file_0 = "data_part_0.parquet"
file_1 = "data/data_part_1.parquet"

df_0 = pd.read_parquet(file_0)
df_1 = pd.read_parquet(file_1)

model_df = pd.concat([df_0, df_1], ignore_index=True)
print(model_df.shape)

(1150000, 16)


In [2]:
model_df.head()

,account_id,month,company_size,industry,contract_type,discount_pct,regime_state,active_users,usage_growth,feature_adoption_rate,error_rate,tickets_count,ticket_growth,payment_delay_flag,current_mrr,next_month_mrr
0,0,1,SMB,Energy,Monthly,0.054167,stable,4,0.017574,0.002766,0.102689,0,0.0,0,112.703497,50.000000
1,0,2,SMB,Energy,Monthly,0.054167,stable,11,0.027244,0.000000,0.102174,2,0.0,1,50.000000,51.903945
2,0,3,SMB,Energy,Monthly,0.054167,decline,12,0.004655,0.082195,0.195298,2,0.0,0,51.903945,50.000000
3,0,4,SMB,Energy,Monthly,0.054167,decline,17,-0.024496,0.062898,0.104958,3,0.5,1,50.000000,55.680067
4,0,5,SMB,Energy,Monthly,0.054167,decline,20,-0.073376,0.120356,0.076803,0,-1.0,0,55.680067,50.000000


In [3]:
overview = pd.DataFrame({
    "dtype": model_df.dtypes,
    "unique_values": model_df.nunique(),
    "missing_values": model_df.isna().sum(),
    "missing_pct": model_df.isna().mean() * 100
})

print(overview)

                         dtype  unique_values  missing_values  missing_pct
account_id               Int64          50000               0          0.0
month                    int64             23               0          0.0
company_size               str              3               0          0.0
industry                   str             10               0          0.0
contract_type              str              2               0          0.0
discount_pct           float64          50001               0          0.0
regime_state               str              3               0          0.0
active_users             int64            785               0          0.0
usage_growth           float64        1150000               0          0.0
feature_adoption_rate  float64        1023028               0          0.0
error_rate             float64        1133822               0          0.0
tickets_count            int64             42               0          0.0
ticket_growth          fl

In [4]:
# Revenue target
model_df["mrr_change"] = (
    model_df["next_month_mrr"] - model_df["current_mrr"]
)

# Churn target
model_df["churn"] = (
    model_df["next_month_mrr"] < model_df["current_mrr"]
).astype(int)

model_df[["current_mrr", "next_month_mrr", "mrr_change", "churn"]].head()

,current_mrr,next_month_mrr,mrr_change,churn
0,112.703497,50.000000,-62.703497,1
1,50.000000,51.903945,1.903945,0
2,51.903945,50.000000,-1.903945,1
3,50.000000,55.680067,5.680067,0
4,55.680067,50.000000,-5.680067,1


In [5]:
model_df["churn"].value_counts(normalize=True)

churn
0    0.671392
1    0.328608
Name: proportion, dtype: float64

# Feature Extraction & Selection

In [6]:
features = [
    "month",
    "company_size",
    "industry",
    "contract_type",
    "discount_pct",
    "regime_state",
    "active_users",
    "usage_growth",
    "feature_adoption_rate",
    "error_rate",
    "tickets_count",
    "ticket_growth",
    "payment_delay_flag",
    "current_mrr"
]

In [7]:
X = model_df[features]

# Targets
y_reg = model_df["next_month_mrr"]
y_cls = model_df["churn"]

# Encoding & Scaling

## 3.1 Train/Test Split

In [8]:
train_df = model_df[model_df["month"] <= 18].copy()
test_df = model_df[model_df["month"] > 18].copy()

In [9]:
X_train = train_df[features]
X_test = test_df[features]

# Regression target
y_train_reg = train_df["next_month_mrr"]
y_test_reg = test_df["next_month_mrr"]

# Classification target
y_train_cls = train_df["churn"]
y_test_cls = test_df["churn"]

## 3.2 Scaled

In [10]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

categorical_features = [
    "company_size",
    "industry",
    "contract_type",
    "regime_state"
]

numeric_features = [
    "month",
    "discount_pct",
    "active_users",
    "usage_growth",
    "feature_adoption_rate",
    "error_rate",
    "tickets_count",
    "ticket_growth",
    "payment_delay_flag",
    "current_mrr"
]

In [11]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            categorical_features
        ),
        (
            "numeric",
            StandardScaler(),
            numeric_features
        )
    ]
)

In [12]:
X_train_encoded = preprocessor.fit_transform(X_train)
X_test_encoded = preprocessor.transform(X_test)

## 3.3 Unscaled

In [13]:
preprocessor_tree = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            categorical_features
        ),
        (
            "numeric",
            "passthrough",
            numeric_features
        )
    ]
)

# Evaluation Function

## 4.1 Regression Function

In [14]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

def evaluate_regression(model, X_test, y_test):
    
    predictions = model.predict(X_test)

    mae = mean_absolute_error(y_test, predictions)
    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    r2 = r2_score(y_test, predictions)

    return {
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    }

## 4.2 Classification

In [15]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

def evaluate_classification(model, X_test, y_test):

    predictions = model.predict(X_test)
    probabilities = model.predict_proba(X_test)[:, 1]

    return {
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(y_test, predictions),
        "Recall": recall_score(y_test, predictions),
        "F1": f1_score(y_test, predictions),
        "ROC-AUC": roc_auc_score(y_test, probabilities)
    }

# Regression Models

## 5.1 Define Models

In [16]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor,
    BaggingRegressor,
    GradientBoostingRegressor,
    AdaBoostRegressor
)

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

## 5.2 Train

### linear regression

In [17]:
from sklearn.linear_model import LinearRegression

linear_model = LinearRegression()
linear_model.fit(X_train_encoded, y_train_reg)
linear_pred = linear_model.predict(X_test_encoded)

linear_metrics = evaluate_regression(
    linear_model,
    X_test_encoded,
    y_test_reg
)

linear_metrics

{'MAE': 198.42472879204894,
 'RMSE': np.float64(1006.0649847577906),
 'R2': 0.8622373259172992}

### decision tree

In [18]:
from sklearn.tree import DecisionTreeRegressor

dt_model = DecisionTreeRegressor(
    max_depth=12,
    random_state=42
)

dt_model.fit(X_train_encoded, y_train_reg)
dt_pred = dt_model.predict(X_test_encoded)

dt_metrics = evaluate_regression(
    dt_model,
    X_test_encoded,
    y_test_reg
)

dt_metrics

{'MAE': 69.68052135591947,
 'RMSE': np.float64(905.464967228245),
 'R2': 0.8884106365770934}

### random forest

In [19]:
from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=15,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train_encoded, y_train_reg)
rf_pred = rf_model.predict(X_test_encoded)

rf_metrics = evaluate_regression(
    rf_model,
    X_test_encoded,
    y_test_reg
)

rf_metrics

{'MAE': 59.87059859386084,
 'RMSE': np.float64(845.3511169117708),
 'R2': 0.902735634558459}

### gradinet boosting

In [20]:
from sklearn.ensemble import GradientBoostingRegressor

gradient_model = GradientBoostingRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=5,
    random_state=42
)

gradient_model.fit(X_train_encoded, y_train_reg)
gradient_pred = gradient_model.predict(X_test_encoded)

gradient_metrics = evaluate_regression(
    gradient_model,
    X_test_encoded,
    y_test_reg
)

gradient_metrics

{'MAE': 62.07753123181078,
 'RMSE': np.float64(857.2172705194424),
 'R2': 0.8999858793869948}

### adaboost

In [21]:
from sklearn.ensemble import AdaBoostRegressor

adaboost_model = AdaBoostRegressor(
    n_estimators=200,
    learning_rate=0.05,
    random_state=42
)

adaboost_model.fit(X_train_encoded, y_train_reg)
adaboost_pred = adaboost_model.predict(X_test_encoded)

adaboost_metrics = evaluate_regression(
    adaboost_model,
    X_test_encoded,
    y_test_reg
)

adaboost_metrics

{'MAE': 195.2695718381241,
 'RMSE': np.float64(952.2986490091286),
 'R2': 0.8765685495057618}

### xgboost

In [22]:
from xgboost import XGBRegressor

xgb_model = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(X_train_encoded, y_train_reg)
xgb_pred = xgb_model.predict(X_test_encoded)

xgb_metrics = evaluate_regression(
    xgb_model,
    X_test_encoded,
    y_test_reg
)

xgb_metrics

{'MAE': 118.85462063661535,
 'RMSE': np.float64(1516.796223757648),
 'R2': 0.686863297643177}

### lightgbm

In [23]:
from lightgbm import LGBMRegressor

lgbm_model = LGBMRegressor(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    n_jobs=-1
)

lgbm_model.fit(X_train_encoded, y_train_reg)
lgbm_pred = lgbm_model.predict(X_test_encoded)

lgbm_metrics = evaluate_regression(
    lgbm_model,
    X_test_encoded,
    y_test_reg
)

lgbm_metrics

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.026404 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1808
[LightGBM] [Info] Number of data points in the train set: 900000, number of used features: 28
[LightGBM] [Info] Start training from score 256.090678


/home/youssef/ml_env/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/youssef/ml_env/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


{'MAE': 145.64348727828968,
 'RMSE': np.float64(1788.0572917817226),
 'R2': 0.5648466249043524}

## 5.3 Comparison

In [24]:
regression_comparison = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Decision Tree",
        "Random Forest",
        "Gradient Boosting",
        "AdaBoost",
        "XGBoost",
        "LightGBM"
    ],
    "MAE": [
        linear_metrics["MAE"],
        dt_metrics["MAE"],
        rf_metrics["MAE"],
        gradient_metrics["MAE"],
        adaboost_metrics["MAE"],
        xgb_metrics["MAE"],
        lgbm_metrics["MAE"]
    ],
    "RMSE": [
        linear_metrics["RMSE"],
        dt_metrics["RMSE"],
        rf_metrics["RMSE"],
        gradient_metrics["RMSE"],
        adaboost_metrics["RMSE"],
        xgb_metrics["RMSE"],
        lgbm_metrics["RMSE"]
    ],
    "R2": [
        linear_metrics["R2"],
        dt_metrics["R2"],
        rf_metrics["R2"],
        gradient_metrics["R2"],
        adaboost_metrics["R2"],
        xgb_metrics["R2"],
        lgbm_metrics["R2"]
    ]
})

regression_comparison.sort_values("MAE")

,Model,MAE,RMSE,R2
2,Random Forest,59.870599,845.351117,0.902736
3,Gradient Boosting,62.077531,857.217271,0.899986
1,Decision Tree,69.680521,905.464967,0.888411
5,XGBoost,118.854621,1516.796224,0.686863
6,LightGBM,145.643487,1788.057292,0.564847
4,AdaBoost,195.269572,952.298649,0.876569
0,Linear Regression,198.424729,1006.064985,0.862237


# Classification Models

## 6.1 Define Models

In [25]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    BaggingClassifier,
    GradientBoostingClassifier,
    AdaBoostClassifier
)

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

## 6.2 Train

### logistic regression

In [26]:
from sklearn.linear_model import LogisticRegression

logistic_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

logistic_model.fit(
    X_train_encoded,
    y_train_cls
)

logistic_pred = logistic_model.predict(
    X_test_encoded
)

logistic_metrics = evaluate_classification(
    logistic_model,
    X_test_encoded,
    y_test_cls
)

logistic_metrics

{'Accuracy': 0.792836,
 'Precision': 0.6725230856967384,
 'Recall': 0.4540890376469891,
 'F1': 0.5421304359357457,
 'ROC-AUC': 0.8017435046556735}

### decision tree

In [27]:
from sklearn.tree import DecisionTreeClassifier

dt_model = DecisionTreeClassifier(
    max_depth=12,
    random_state=42
)

dt_model.fit(
    X_train_encoded,
    y_train_cls
)

dt_pred = dt_model.predict(
    X_test_encoded
)

dt_metrics = evaluate_classification(
    dt_model,
    X_test_encoded,
    y_test_cls
)

dt_metrics

{'Accuracy': 0.807084,
 'Precision': 0.7674753916539582,
 'Recall': 0.40992565386096386,
 'F1': 0.5344106885999208,
 'ROC-AUC': 0.8397258565203709}

### random forest

In [28]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(
    X_train_encoded,
    y_train_cls
)

rf_pred = rf_model.predict(
    X_test_encoded
)

rf_metrics = evaluate_classification(
    rf_model,
    X_test_encoded,
    y_test_cls
)

rf_metrics

{'Accuracy': 0.80396,
 'Precision': 0.8078355727018758,
 'Recall': 0.3597346050176239,
 'F1': 0.49779690542063737,
 'ROC-AUC': 0.8550130141370296}

### gradient boosting

In [29]:
from sklearn.ensemble import GradientBoostingClassifier

gradient_model = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=5,
    random_state=42
)

gradient_model.fit(
    X_train_encoded,
    y_train_cls
)

gradient_pred = gradient_model.predict(
    X_test_encoded
)

gradient_metrics = evaluate_classification(
    gradient_model,
    X_test_encoded,
    y_test_cls
)

gradient_metrics

{'Accuracy': 0.811868,
 'Precision': 0.7872585032106104,
 'Recall': 0.4158052190397204,
 'F1': 0.5441876241701797,
 'ROC-AUC': 0.8603412947279168}

### adaboost

In [30]:
from sklearn.ensemble import AdaBoostClassifier

adaboost_model = AdaBoostClassifier(
    n_estimators=200,
    learning_rate=0.05,
    random_state=42
)

adaboost_model.fit(
    X_train_encoded,
    y_train_cls
)

adaboost_pred = adaboost_model.predict(
    X_test_encoded
)

adaboost_metrics = evaluate_classification(
    adaboost_model,
    X_test_encoded,
    y_test_cls
)

adaboost_metrics

{'Accuracy': 0.778148,
 'Precision': 0.7393893675308691,
 'Recall': 0.2758064038387488,
 'F1': 0.4017517177404567,
 'ROC-AUC': 0.7315231821334489}

### xgboost

In [31]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    eval_metric="logloss"
)

xgb_model.fit(
    X_train_encoded,
    y_train_cls
)

xgb_pred = xgb_model.predict(
    X_test_encoded
)

xgb_metrics = evaluate_classification(
    xgb_model,
    X_test_encoded,
    y_test_cls
)

xgb_metrics

{'Accuracy': 0.812864,
 'Precision': 0.7851760176017601,
 'Recall': 0.4228103432955185,
 'F1': 0.5496428640187906,
 'ROC-AUC': 0.8628850322885572}

### lightgbm

In [32]:
from lightgbm import LGBMClassifier

lgbm_model = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    n_jobs=-1
)

lgbm_model.fit(
    X_train_encoded,
    y_train_cls
)

lgbm_pred = lgbm_model.predict(
    X_test_encoded
)

lgbm_metrics = evaluate_classification(
    lgbm_model,
    X_test_encoded,
    y_test_cls
)

lgbm_metrics

[LightGBM] [Info] Number of positive: 310377, number of negative: 589623
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.022467 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1808
[LightGBM] [Info] Number of data points in the train set: 900000, number of used features: 28
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.344863 -> initscore=-0.641696
[LightGBM] [Info] Start training from score -0.641696


/home/youssef/ml_env/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/youssef/ml_env/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/youssef/ml_env/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


{'Accuracy': 0.813704,
 'Precision': 0.7816499946219211,
 'Recall': 0.4304967269926839,
 'F1': 0.5552096265877184,
 'ROC-AUC': 0.8632773433300951}

## 6.3 Comparison

In [33]:
classification_results = pd.DataFrame({

    "Model": [
        "Logistic Regression",
        "Decision Tree",
        "Random Forest",
        "Gradient Boosting",
        "AdaBoost",
        "XGBoost",
        "LightGBM"
    ],

    "Accuracy": [
        logistic_metrics["Accuracy"],
        dt_metrics["Accuracy"],
        rf_metrics["Accuracy"],
        gradient_metrics["Accuracy"],
        adaboost_metrics["Accuracy"],
        xgb_metrics["Accuracy"],
        lgbm_metrics["Accuracy"]
    ],

    "Precision": [
        logistic_metrics["Precision"],
        dt_metrics["Precision"],
        rf_metrics["Precision"],
        gradient_metrics["Precision"],
        adaboost_metrics["Precision"],
        xgb_metrics["Precision"],
        lgbm_metrics["Precision"]
    ],

    "Recall": [
        logistic_metrics["Recall"],
        dt_metrics["Recall"],
        rf_metrics["Recall"],
        gradient_metrics["Recall"],
        adaboost_metrics["Recall"],
        xgb_metrics["Recall"],
        lgbm_metrics["Recall"]
    ],

    "F1": [
        logistic_metrics["F1"],
        dt_metrics["F1"],
        rf_metrics["F1"],
        gradient_metrics["F1"],
        adaboost_metrics["F1"],
        xgb_metrics["F1"],
        lgbm_metrics["F1"]
    ],

    "ROC-AUC": [
        logistic_metrics["ROC-AUC"],
        dt_metrics["ROC-AUC"],
        rf_metrics["ROC-AUC"],
        gradient_metrics["ROC-AUC"],
        adaboost_metrics["ROC-AUC"],
        xgb_metrics["ROC-AUC"],
        lgbm_metrics["ROC-AUC"]
    ]
})

classification_results = classification_results.sort_values(
    by="ROC-AUC",
    ascending=False
).reset_index(drop=True)

classification_results

,Model,Accuracy,Precision,Recall,F1,ROC-AUC
0,LightGBM,0.813704,0.781650,0.430497,0.555210,0.863277
1,XGBoost,0.812864,0.785176,0.422810,0.549643,0.862885
2,Gradient Boosting,0.811868,0.787259,0.415805,0.544188,0.860341
3,Random Forest,0.803960,0.807836,0.359735,0.497797,0.855013
4,Decision Tree,0.807084,0.767475,0.409926,0.534411,0.839726
5,Logistic Regression,0.792836,0.672523,0.454089,0.542130,0.801744
6,AdaBoost,0.778148,0.739389,0.275806,0.401752,0.731523


# Save Model

In [34]:
import joblib

# Final models
final_regression_model = rf_model
final_classification_model = lgbm_model

In [ ]:
# Save models
joblib.dump(
    final_regression_model,
    "models/final_mrr_model.pkl"
)

joblib.dump(
    final_classification_model,
    "models/final_churn_model.pkl"
)

['/home/youssef/Desktop/Data_Projects/saas_revenue_app/models/final_churn_model.pkl']

In [ ]:
# Save preprocessor
joblib.dump(
    preprocessor,
    "models/preprocessor.pkl"
)

print("Final models and preprocessor saved successfully.")

Final models and preprocessor saved successfully.
